In [2]:
import torch
from PIL import Image
from transformers import XLMRobertaTokenizer
from torchvision import transforms
from tqdm import tqdm
import pandas as pd
import os
import json
from collections import Counter

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [8]:
# # concat imag_path

# df_1 = pd.read_csv("/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/vqa/vqa_rephrased_by_llama.csv")
# df_2 = pd.read_csv("/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv")
# df_img_path = df_2[['img_path','ID']]

# df_merge =  pd.merge(df_1, df_img_path, on='ID', how='left')

# df_merge

# df_merge.to_csv("/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/vqa/vqa_rephrased_by_llama_with_img_path.csv", index=False)

In [ ]:
# --- 1. 모델 구조 import ---
try:
    from modeling_finetune import beit3_large_patch16_768_vqav2
except ImportError:
    print("오류: 'modeling_finetune.py'를 찾을 수 없습니다.")
    print("BEiT-3 레포지토리에서 해당 파일을 다운로드하여 같은 디렉토리에 위치시키거나, sys.path에 경로를 추가해주세요.")
    exit()

# --- 2. 경로 설정 (사용자 환경에 맞게 수정) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

# BEiT-3 모델 관련 경로
beit3_model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3_large_indomain_patch16_768_vgqaaug_vqa.pth"
spm_model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3.spm"
vqa_answer_label_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/v2_mscoco_train2014_annotations.json"

# 데이터 관련 경로
preprocessed_csv_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/caption/test_preprocessed_by_llm.csv'
image_dir = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/'
submission_csv_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/submission/submission_answer.csv'
sample_submission_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/sample_submission.csv'


# --- 3. BEiT-3 모델 및 토크나이저 로드 ---
print("VQA 모델 (BEiT-3)을 로딩합니다...")
beit3_tokenizer = XLMRobertaTokenizer(spm_model_path)
beit3_model = beit3_large_patch16_768_vqav2()

try:
    checkpoint = torch.load(beit3_model_path, map_location='cpu')
    beit3_model.load_state_dict(checkpoint['model'])
    beit3_model.to(device)
    beit3_model.eval()
    print("BEiT-3 모델 로딩 완료.")
except FileNotFoundError:
    print(f"오류: BEiT-3 모델 가중치 파일을 찾을 수 없습니다. 경로를 확인하세요: {beit3_model_path}")
    exit()


# --- 4. VQAv2 답변 사전 생성 및 'yes'/'no' 인덱스 확보 ---
print("VQAv2 답변 사전을 생성합니다...")
try:
    with open(vqa_answer_label_path, 'r') as f:
        vqa_data = json.load(f)
    raw_annotations = vqa_data['annotations']
    answer_counter = Counter(ann['multiple_choice_answer'] for ann in raw_annotations)
    top_answers = answer_counter.most_common(3129)
    answer_vocab = [answer for answer, count in top_answers]
    answer_to_idx = {answer: i for i, answer in enumerate(answer_vocab)}

    if 'yes' in answer_to_idx and 'no' in answer_to_idx:
        yes_idx = answer_to_idx['yes']
        no_idx = answer_to_idx['no']
        print(f"답변 사전 생성 완료. 'yes' 인덱스: {yes_idx}, 'no' 인덱스: {no_idx}")
    else:
        raise ValueError("'yes' 또는 'no'를 답변 사전에서 찾을 수 없습니다!")
except Exception as e:
    print(f"답변 사전 생성 중 오류 발생: {e}")
    exit()


# --- 5. 데이터 로드 및 추론 준비 ---
try:
    df = pd.read_csv(preprocessed_csv_path)
    print(f"전처리된 데이터 '{preprocessed_csv_path}'에서 {len(df)}개의 행을 읽었습니다.")
except FileNotFoundError:
    print(f"오류: 전처리된 CSV 파일을 찾을 수 없습니다. 경로를 확인하세요: {preprocessed_csv_path}")
    exit()

submission_results = []
transform = transforms.Compose([
    transforms.Resize((768, 768)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])


# --- 6. 하이브리드 로직 기반 추론 실행 ---
print("BEiT-3 추론을 시작합니다...")
for index, row in tqdm(df.iterrows(), total=len(df), desc="BEiT-3 Inference"):
    target_answer = row.get('target_answer', 'yes')
    target_idx = yes_idx if target_answer == 'yes' else no_idx
    
    image_path = os.path.join(image_dir, row['img_path'].lstrip('./'))
    if not os.path.exists(image_path):
        submission_results.append({'ID': row['ID'], 'answer': 'A'})
        continue
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
    except Exception as e:
        submission_results.append({'ID': row['ID'], 'answer': 'A'})
        continue

    choice_scores = {}
    options = ['A', 'B', 'C', 'D']
    for choice_key in options:
        caption = row[f'caption_{choice_key}']
        
        if not isinstance(caption, str) or not caption:
            choice_scores[choice_key] = -float('inf')
            continue

        encoding = beit3_tokenizer(
            caption, padding='max_length', truncation=True,
            max_length=40, return_tensors='pt'
        )
        question_tokens = encoding['input_ids'].to(device)
        padding_mask = encoding['attention_mask'].to(device)

        with torch.no_grad():
            output_logits = beit3_model(
                image=image_tensor,
                question=question_tokens,
                padding_mask=padding_mask
            )
        
        target_score = output_logits[0, target_idx].item()
        choice_scores[choice_key] = target_score

    if choice_scores:
        # 점수가 가장 높은 보기의 키 ('A', 'B', 'C', 'D')를 찾음
        best_choice_key = max(choice_scores, key=choice_scores.get)
        
        # ★★★ 이 부분이 수정되었습니다 ★★★
        # 답변의 '내용'이 아닌 '키(A,B,C,D)' 자체를 저장합니다.
        predicted_answer = best_choice_key
    else:
        predicted_answer = 'A'

    submission_results.append({'ID': row['ID'], 'answer': predicted_answer})

# --- 7. 최종 결과 제출 파일로 저장 ---
print("추론이 완료되었습니다. 제출 파일을 생성합니다.")

submission_df = pd.DataFrame(submission_results)

try:
    sample_df = pd.read_csv(sample_submission_path)
    submission_df = submission_df.set_index('ID').loc[sample_df['ID']].reset_index()
except FileNotFoundError:
    print(f"Warning: '{sample_submission_path}'를 찾을 수 없어 ID 순서 정렬을 건너뜁니다.")

submission_df.to_csv(submission_csv_path, index=False)

print(f"\n✅ 모든 작업 완료! 최종 제출 파일이 '{submission_csv_path}'에 저장되었습니다.")
print("\n--- 제출 파일 미리보기 (상위 5개) ---")
print(submission_df.head())

사용 장치: cuda
VQA 모델 (BEiT-3)을 로딩합니다...
BEiT-3 모델 로딩 완료.
VQAv2 답변 사전을 생성합니다...
답변 사전 생성 완료. 'yes' 인덱스: 0, 'no' 인덱스: 1
전처리된 데이터 '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/caption/test_preprocessed_by_llm.csv'에서 60개의 행을 읽었습니다.
BEiT-3 추론을 시작합니다...


BEiT-3 Inference: 100%|██████████| 60/60 [00:38<00:00,  1.55it/s]

추론이 완료되었습니다. 제출 파일을 생성합니다.

✅ 모든 작업 완료! 최종 제출 파일이 '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/submission/submission_answer.csv'에 저장되었습니다.

--- 제출 파일 미리보기 (상위 5개) ---
         ID answer
0  TEST_000      A
1  TEST_001      A
2  TEST_002      A
3  TEST_003      B
4  TEST_004      B


: 